<a href="https://colab.research.google.com/github/corbinwhcurtin/smart-finance-assistant/blob/main/starter_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Project Overview

Welcome to your **Smart Finance Assistant** development journey! This notebook will evolve from basic CSV processing to a complete AI-powered finance application.

**Final Application Components:**
- 💬 **AI Chat Interface** - Financial advice personality
- 📊 **Data Analysis** - CSV transaction processing  
- 🔍 **RAG System** - Retrieval from financial documents
- 🛠️ **Custom Tools** - Calculators and utilities
- 🌐 **Gradio UI** - Professional web interface

**Development Approach**: Build progressively from foundation to advanced features, using AI collaboration throughout.

---

# 🚀 Getting Started: Foundation Setup

## Initial Setup
This cell installs the necessary libraries. In a Colab environment, you would uncomment the first line.

In [78]:
# ── Cell 1: Setup & Imports ───────────────────────────────────
!pip install gradio pandas hands-on-ai plotly -q

import os
import json
import calendar
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly
import gradio as gr

from datetime import datetime, date
from pathlib import Path
from typing import Optional

warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")
print(f"  Pandas:  {pd.__version__}")
print(f"  Plotly:  {plotly.__version__}")
print(f"  Gradio:  {gr.__version__}")

Libraries loaded successfully.
  Pandas:  2.2.2
  Plotly:  5.24.1
  Gradio:  5.50.0


## Hands-on-AI Configuration

Set up the hands-on-ai package for advanced features (chat, RAG, tools):

In [79]:
# ── Cell 2: AI Configuration ──────────────────────────────────
from hands_on_ai.chat import get_response
from hands_on_ai import agent, rag

import os
os.environ['HANDS_ON_AI_SERVER']  = 'https://ollama.locollm.org'
os.environ['HANDS_ON_AI_MODEL']   = 'gemma3:4b'
os.environ['HANDS_ON_AI_API_KEY'] = 'Curtin2026ISYS20015002'

# Test connection
try:
    test = get_response("Reply with exactly three words: connection is working")
    print("AI connection confirmed.")
    print(f"  Server:   {os.environ['HANDS_ON_AI_SERVER']}")
    print(f"  Model:    {os.environ['HANDS_ON_AI_MODEL']}")
    print(f"  Response: {test}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Check your network connection and API key.")

AI connection confirmed.
  Server:   https://ollama.locollm.org
  Model:    gemma3:4b
  Response: Connection is working


The API key this semester is:  **isys2001-assignment-key**

## Connection Test

Test that everything is working correctly:

In [80]:
# Test the connection to the hands-on-ai server
try:
    response = get_response("Hello! I'm building a Smart Finance Assistant.")
    print("✅ Hands-on-AI connection successful!")
    print(f"Response: {response}")
except Exception as e:
    print(f"❌ Connection issue: {e}")
    print("You can still work on the data processing foundation without this.")

✅ Hands-on-AI connection successful!
Response: That's fantastic! Building a Smart Finance Assistant is a really exciting and useful project. I'm happy to help in any way I can. 

To get us started, could you tell me a little more about what you're building? For example:

*   **What’s the overall goal of your assistant?** (e.g., budgeting, investment tracking, debt management, bill reminders, financial advice?)
*   **What platforms are you targeting?** (e.g., mobile app, web app, voice assistant integration like Alexa or Google Assistant?)
*   **What kind of features are you planning?** (e.g., transaction categorization, goal setting, spending reports, financial calculations?)
*   **What's your current level of development?** (e.g., just an idea, have you started coding, do you need help with a specific task?)

Don't worry if you don't have all the answers yet. Just tell me what you're working on and where you think you might need assistance. 

Let's brainstorm and see how I can be a us

In [83]:
#Data Persistence (Save / Load Function)

SAVE_FILE = "finance_data.json"

def save_app_data(app_state: dict) -> None:
    """
    Serialises and saves the full app state to a JSON file.
    Called automatically whenever data changes.
    """
    try:
        serialisable = {
            "budget_plan":      app_state.get("budget_plan"),
            "safety_net":       app_state.get("safety_net"),
            "savings_goals":    app_state.get("savings_goals", []),
            "monthly_snapshots": []
        }

        # serialise each monthly snapshot
        for snapshot in app_state.get("monthly_snapshots", []):
            s = snapshot.copy()
            # convert category_summary DataFrame to dict
            if isinstance(s.get("category_summary"), pd.DataFrame):
                s["category_summary"] = s["category_summary"].to_dict()
            serialisable["monthly_snapshots"].append(s)

        with open(SAVE_FILE, "w") as f:
            json.dump(serialisable, f, indent=2, default=str)

    except Exception as e:
        print(f"Auto-save failed: {e}")


def load_app_data() -> dict:
    """
    Loads saved app state from JSON file if it exists.
    Returns empty state if no save file found.
    """
    empty_state = {
        "budget_plan":       None,
        "safety_net":        None,
        "savings_goals":     [],
        "monthly_snapshots": []
    }

    if not Path(SAVE_FILE).exists():
        print("No save file found. Starting fresh.")
        return empty_state

    try:
        with open(SAVE_FILE, "r") as f:
            data = json.load(f)

        # restore category_summary DataFrames
        for snapshot in data.get("monthly_snapshots", []):
            if isinstance(snapshot.get("category_summary"), dict):
                snapshot["category_summary"] = pd.DataFrame.from_dict(
                    snapshot["category_summary"]
                )

        print(f"Save file loaded. {len(data.get('monthly_snapshots', []))} monthly snapshot(s) found.")
        return data

    except Exception as e:
        print(f"Could not load save file: {e}")
        return empty_state


def get_snapshot_label(df: pd.DataFrame) -> str:
    """
    Generates a month/year label from a transaction DataFrame.
    e.g. 'March 2025'
    """
    if df is None or df.empty:
        return "Unknown"
    earliest = df['date'].min()
    latest   = df['date'].max()
    if earliest.month == latest.month:
        return earliest.strftime("%B %Y")
    return f"{earliest.strftime('%b')} – {latest.strftime('%b %Y')}"


# ── Initialise global app state ───────────────────────────────
app_state = load_app_data()
app_state["current_df"]       = None   # active transaction DataFrame
app_state["current_analysis"] = None   # active analysis dictionary
app_state["current_label"]    = None   # e.g. "March 2025"

print(f"App state initialised.")
print(f"  Budget plan:   {'Set' if app_state['budget_plan'] else 'Not set'}")
print(f"  Safety net:    {'Set' if app_state['safety_net'] else 'Not set'}")
print(f"  Savings goals: {len(app_state['savings_goals'])}")
print(f"  Past months:   {len(app_state['monthly_snapshots'])}")

No save file found. Starting fresh.
App state initialised.
  Budget plan:   Not set
  Safety net:    Not set
  Savings goals: 0
  Past months:   0


# 🏗️ Foundation: Data Processing Skills

Before building advanced features, establish solid data processing foundations. This section focuses on CSV transaction analysis - the core of your finance assistant.

## Foundation Skill Checkpoint ✅

**Master these basics before advancing to chat/RAG/tools:**
- [ ] Load and clean CSV transaction data
- [ ] Handle real-world data issues (dollar signs, missing values)
- [ ] Calculate spending summaries by category  
- [ ] Generate business-appropriate insights
- [ ] Format output for professional presentation
- [ ] Test functions with various data scenarios

::: {.callout-tip}
## 🤖 AI Collaboration Strategy

For this foundation work, use AI to:
1. **Generate initial code** with specific business context
2. **Handle data cleaning** and validation
3. **Create professional formatting** for outputs
4. **Suggest business insights** from data patterns
5. **Help with testing** edge cases and error handling

**Remember**: You're directing AI like a junior developer - always review and improve their suggestions!
:::

## Sample Transaction Data Setup

Create or load sample transaction data to work with:

In [ ]:
# 🤖 AI Collaboration Opportunity:
# Ask AI to help you create realistic sample transaction data
# Include Australian businesses and various spending categories

# Sample data structure for testing
sample_transactions = {
    'Date': ['2024-08-01', '2024-08-02', '2024-08-03', '2024-08-04', '2024-08-05'],
    'Amount': ['$45.50', '$12.00', '$89.95', '$3.50', '-$25.00'],  # Note: includes $ signs and refund
    'Category': ['Groceries', 'Transport', 'Entertainment', 'Coffee', 'Refund'],
    'Description': ['Woolworths', 'Opal Card', 'Concert Tickets', 'Campus Cafe', 'Returned Item']
}

# Create DataFrame from sample data
df_sample = pd.DataFrame(sample_transactions)
print("📋 Sample transaction data created:")
print(df_sample)

---

# 📊 Six-Step Development Methodology

Your notebook must demonstrate the six-step methodology with clear evidence of AI collaboration at each step.

## STEP 1: Understand the Problem

**🎯 Define Your Finance Problem**

In this section, clearly state your chosen finance problem in business terms.

::: {.callout-note}
## Problem Definition Template


I want to help university students understand and alalyse their spending habits from their bank transations (CSV). Focus should be on reducing unnessisary spending during semester but budget for saving and social spending money. It should also provide feedback and insights into were money can "disapear"


AI Prompt

Help me brainstorm features than a csv analysis tool that is useful for university students budgeting while allowing social spending and savings focusing on reducing money that disapears

AI Reply of Features:
1. Catching invisible spending
2. Making a budget
3. Allowing social spending
4. Spending pattern insight
5. Motivation and goal tracking

**Your Problem Statement:**

University Student often struggle budgeting while studying full-time. The goal of the problem is to set budget and analyse spending with a focus of sticking to a budget while still allowing social spending. Using CSV bank transaction data to reduce small transaction that are unnecessary and add up. The program should help students understand their spending habits and provide realistic budget while saving money at the same time.

## STEP 2: Identify Inputs and Outputs

**📥 Define Your Data Flow**

Data inputs:
-	Bank Transaction CSV files with date, description, amount, description
-	Manually enter cash transactions
-	Manually overside incorrect transaction category
-	Monthly budgets, saving goals amount

Processing
-	Clean data to match format (e.g. missing $)
-	Auto categorise transactions
-	Detect reoccurring payments e.g. coffee every day
-	Separate essentials vs other spending e.g. rent vs pub

Insights
-	Essential vs choice spending
-	Invisible spending e.g. small daily payments that can be reduced
-	Subscriptions
-	Daily/weekly allowable spending
-	Social spending amount (is it within reason/% of budget)
-	Where to cut spending
-	Day of week/ payday spending spikes
-	Savings goal progress
-	Financial health score e.g. safety savings bucket
-	AI generated spending avoid in easy to understand English


## Input/Output Analysis Template

Inputs:
CSV file with transaction data (columns: Date, Description, Amount, and optionally Category)
Manually entered transactions (date, description, amount, category) for cash and non-bank spending
Essential flags — user-marked transactions or merchants that cannot be reduced (rent, power, internet)
Category corrections — user edits to fix auto-categorisation errors
Monthly discretionary budget (user-defined)
Savings goal (target amount + target date e.g. "Bali trip — $2,000 by December")
Time period for analysis (e.g. last 30 days, last 3 months)

Outputs:
Essential vs discretionary spend breakdown — shows committed costs before any choices are made
Spending summary by category with totals, averages, and transaction counts
Invisible spend report — small frequent transactions (e.g. coffees, snacks) aggregated and flagged
Subscription audit — recurring charges listed with estimated monthly and annual cost
Day-of-week and post-payday spike analysis — when overspending tends to happen
Social spending health check — social budget as a percentage of discretionary spend, framed positively
Daily discretionary allowance — how much remains per day for the rest of the month
"Cut X → save Y" statements — trade-off insights targeting only reducible spend
Savings goal progress tracker — current progress, weekly contribution needed, projected completion date
Financial health score (0–100) — based on discretionary budget adherence and savings rate
AI-generated personalised advice summary via get_response() — plain-English recommendations grounded in the user's actual reducible spending



## STEP 3: Work the Problem by Hand

**✋ Manual Calculation Examples**

Show 2-3 worked examples to understand the logic before coding.


## Example Business Calculation

Given this sample data:

| Date | Amount | Category | Description |
|------|--------|----------|-------------|
| 2024-08-01 | $45.50 | Groceries | Woolworths |
| 2024-08-02 | $12.00 | Transport | Opal Card |
| 2024-08-03 | $89.95 | Entertainment | Concert |
| 2024-08-04 | -$15.00 | Refund | Returned item |

**Manual Calculations:**
- Total Spending: $45.50 + $12.00 + $89.95 - $15.00 = $132.45
- By Category: Groceries $45.50, Transport $12.00, Entertainment $89.95
- Average Transaction: $132.45 ÷ 4 = $33.11
- Insight: Entertainment represents 67% of positive spending

**🤖 AI Prompt to Try:**


**Your Manual Examples:**
Scenario 1 — Normal weekly spending (typical uni student, Perth)
Date, Description, Amount, Category
2024-03-01, Coles Supermarket Karrinyup, -87.43, Food & Groceries
2024-03-02, Transperth Smartrider Topup, -20.00, Transport
2024-03-03, Boost Juice Hay St, -8.50, Coffee & Snacks
2024-03-04, Spotify Premium, -11.99, Subscriptions
2024-03-04, McDonald's Northbridge, -13.20, Eating Out
2024-03-05, Instagram Boost Meta, -15.00, Other
2024-03-06, Hungry Jacks Murray St, -11.40, Eating Out
2024-03-07, 7-Eleven Beaufort St, -6.30, Coffee & Snacks
2024-03-07, Casual Work Payment, +280.00, Income

Total spending: 87.43 + 20 + 8.50 + 11.99 + 13.20 + 15.00 + 11.40 + 6.30 = 173.82
Total Income: $280.00
By Category: Food & Groceries $87.43, Transport $20.00, Coffee & Snacks $14.80, Subscriptions $11.99, Eating Out $24.60, Other $15.00
Average Transaction (expenses only): $173.82 ÷ 8 = $21.73
Insight: Eating Out ($24.60) and Coffee & Snacks ($14.80) combined represent 22.7% of spending despite being small individual purchases

## STEP 4: Write Pseudocode

**📝 Plan Your Solution Logic**

Sketch the algorithm in plain English before coding.


**Your Pseudocode:**
Smart Finance Assistant — Pseudocode

FUNCTION: Load and Clean Data

Ask the user to upload their bank CSV file
Check the file has at least a date, description, and amount column
Tidy up the data by fixing date formats, removing dollar signs, and deleting any blank or duplicate rows
If the user has added any cash transactions manually, add those into the same list
Hand the cleaned list of transactions to the next step


FUNCTION: Categorise Transactions

Look at the description of each transaction
Compare it against a list of known merchant names and keywords
Assign the most appropriate category such as Food, Transport, or Social
If the transaction already has a category, leave it as is
If nothing matches, label it as Other
Hand the categorised list to the next step


FUNCTION: Flag Essential Transactions

Automatically mark known essential merchants such as rent, power, and phone bills as essential
Show the user a table of all transactions so they can review the categories and essential flags
Allow the user to correct any category that was wrong
Allow the user to mark or unmark any transaction as essential
Whenever the user makes a change, update all the analysis immediately


FUNCTION: Analyse Spending

Separate transactions into three buckets — income, essential expenses, and discretionary expenses
Calculate the total for each bucket
Subtract any refunds from the relevant categories so they are not double counted
For each discretionary category work out the total spent, number of transactions, and average transaction size
Work out how much of the monthly budget has been used and how much is left per day for the rest of the month
Look for invisible spending by finding small purchases under fifteen dollars that happen repeatedly in the same week
Look for subscriptions by finding the same merchant charging a similar amount every month
Return a summary of all these figures


FUNCTION: Detect Spending Patterns

Work out which day of the week the student tends to spend the most
Check if spending jumps noticeably in the days immediately after payday
Look at social spending as a percentage of total discretionary spend and give it a positive label such as healthy balance or worth keeping an eye on
Return these pattern observations


FUNCTION: Track Savings Goal

Take the goal name, target amount, and target date from the user
Work out how much has been saved so far based on income minus all expenses
Calculate what percentage of the goal has been reached
Work out how much needs to be set aside each week to hit the goal on time
For each discretionary category suggest how many weeks sooner the goal could be reached if that category was reduced by half
Flag the goal as at risk if the required weekly saving is unrealistically high
Return the goal progress and suggestions


FUNCTION: Calculate Financial Health Score

Start with a perfect score of 100
Reduce the score if spending has gone over the monthly budget
Reduce the score if invisible spending makes up a large portion of discretionary spend
Reduce the score if subscriptions are taking up too much of the budget
Reduce the score if very little is being saved relative to income
Reduce the score slightly if a payday spending spike was detected
Add points back if the savings goal is on track
Keep the final score between 0 and 100
Assign a plain English label such as On Track, Needs Attention, At Risk, or Action Required
Return the score and label


FUNCTION: Generate AI Advice

Gather a plain English summary of the student's finances including their top spending categories, invisible spend total, subscription costs, savings progress, and health score
Send this summary to the AI along with instructions to be encouraging and non-judgmental
Tell the AI to never suggest cutting essential spending and to always protect a reasonable social budget
Tell the AI to focus first on invisible spend and unused subscriptions
Display the AI's personalised recommendations to the user
Allow the user to ask follow up questions in a chat interface


FUNCTION: Build the Interface

Create a simple four tab interface using Gradio
Tab one lets the user upload their CSV, add manual cash transactions, and edit the transaction table
Tab two shows a spending dashboard with category totals, charts, invisible spend, and subscription alerts
Tab three shows the budget tracker, daily allowance, savings goal progress, and cut X save Y suggestions
Tab four shows the financial health score and the AI advice chatbot
Whenever any data changes in any tab, recalculate everything and refresh all the displays


MAIN PROGRAM

Connect to the AI server
Launch the Gradio interface
Wait for the user to upload their data and interact with the tool
Route each user action to the correct function
Display the results

## STEP 5: Convert to Python

**💻 Implementation with AI Collaboration**

Now implement your solution using AI assistance. Focus on creating professional, business-appropriate code.


## 🤖 Implementation Strategy

**Effective AI Prompts for Implementation:**
```
"I'm implementing a Smart Finance Assistant. Based on my pseudocode, please create
a Python function that [specific functionality]. The code should:
- Handle real-world CSV data issues (dollar signs, missing values)
- Include clear comments explaining business logic
- Use professional variable names
- Format output for business presentation
- Include basic error handling"
```

**Remember to critique and improve AI responses before using them!**
:::

### Foundation Data Processing Functions

In [85]:
# ── Cell 4: Data Loading, Cleaning & Categorisation ──────────

CATEGORIES = [
    "Food & Groceries", "Coffee & Snacks", "Eating Out",
    "Transport", "Social", "Subscriptions", "Shopping",
    "Bills & Utilities", "Other", "Income"
]

KEYWORD_MAP = {
    "Food & Groceries":  ["coles", "woolworths", "aldi", "iga", "foodland",
                          "spudshed", "fresh provisions", "market", "deli"],
    "Coffee & Snacks":   ["boost juice", "7-eleven", "starbucks", "gloria jeans",
                          "cafe", "coffee", "muffin break", "chatime", "gong cha"],
    "Eating Out":        ["mcdonald", "hungry jacks", "kfc", "subway", "nando",
                          "domino", "pizza", "uber eats", "doordash", "menulog",
                          "grill'd", "zambreros", "oporto", "red rooster"],
    "Transport":         ["transperth", "smartrider", "uber", "ola", "didi",
                          "ampol", "bp", "caltex", "shell", "puma energy",
                          "parking", "curtin parking"],
    "Subscriptions":     ["spotify", "netflix", "disney", "youtube", "apple",
                          "amazon prime", "binge", "stan", "paramount",
                          "adobe", "microsoft", "google one"],
    "Bills & Utilities": ["rent", "realmark", "ray white", "lj hooker",
                          "property", "synergy", "alinta", "atco", "water",
                          "optus", "telstra", "vodafone", "tpg", "iinet",
                          "insurance", "medicare", "nib", "bupa"],
    "Shopping":          ["kmart", "target", "big w", "jb hi-fi", "harvey norman",
                          "cotton on", "asos", "uniqlo", "h&m", "myer", "david jones",
                          "ebay", "amazon", "etsy", "chemist warehouse"],
    "Social":            ["hotel", "bar", "pub", "tavern", "nightclub", "club",
                          "grosvenor", "rechabite", "amplifier", "rosie",
                          "lucky chan", "the bird", "fomo", "concert", "ticketek",
                          "eventbrite", "bowling", "hoyts", "event cinemas"],
    "Income":            ["salary", "payroll", "wage", "casual work", "pay",
                          "centrelink", "youth allowance", "austudy",
                          "scholarship", "transfer in", "deposit"]
}

ESSENTIAL_KEYWORDS = [
    "rent", "realmark", "ray white", "lj hooker", "property",
    "synergy", "alinta", "atco", "water corporation",
    "optus", "telstra", "vodafone", "tpg", "iinet",
    "insurance", "medicare", "hospital"
]


def load_and_clean_transaction_data(file_path):
    """
    Loads and cleans bank transaction CSV data.
    Handles Australian bank export formats.

    Parameters:
        file_path: path string, Path object, or existing DataFrame
    Returns:
        Cleaned DataFrame or None if loading fails
    """

    # ── Step 1: Load ──────────────────────────────────────────
    try:
        if isinstance(file_path, pd.DataFrame):
            df = file_path.copy()
        else:
            df = pd.read_csv(file_path)
        print(f"Loaded {len(df)} rows.")
    except FileNotFoundError:
        print("File not found. Check the file path and try again.")
        return None
    except Exception as e:
        print(f"Could not read file: {e}")
        return None

    # ── Step 2: Standardise column names ─────────────────────
    df.columns = df.columns.str.strip().str.lower()
    rename_map = {
        "transaction date": "date", "trans date": "date",
        "debit":            "amount", "credit": "amount",
        "merchant":         "description", "details": "description",
        "type":             "category", "memo": "description"
    }
    df = df.rename(columns={
        c: rename_map[c] for c in df.columns if c in rename_map
    })

    # ── Step 3: Validate required columns ────────────────────
    required = ["date", "amount", "description"]
    missing  = [c for c in required if c not in df.columns]
    if missing:
        print(f"Missing required columns: {missing}")
        print(f"Your file has: {list(df.columns)}")
        return None

    # ── Step 4: Clean Amount ──────────────────────────────────
    if df["amount"].dtype == object:
        df["amount"] = (df["amount"]
                        .astype(str)
                        .str.replace("$", "", regex=False)
                        .str.replace(",", "", regex=False)
                        .str.replace("+", "", regex=False)
                        .str.strip())
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    invalid = df["amount"].isna().sum()
    if invalid:
        print(f"  {invalid} rows removed — unreadable amounts.")
    df = df.dropna(subset=["amount"])
    df = df[df["amount"] != 0]

    # ── Step 5: Clean Date ────────────────────────────────────
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
    invalid = df["date"].isna().sum()
    if invalid:
        print(f"  {invalid} rows removed — unreadable dates.")
    df = df.dropna(subset=["date"])

    # ── Step 6: Clean Description ─────────────────────────────
    df["description"] = (df["description"]
                         .astype(str)
                         .str.strip()
                         .str.upper())
    df = df[~df["description"].isin(["", "NAN"])]

    # ── Step 7: Add missing columns ──────────────────────────
    if "category" not in df.columns:
        df["category"] = "Uncategorised"
    if "essential" not in df.columns:
        df["essential"] = False

    # ── Step 8: Remove duplicates ─────────────────────────────
    before = len(df)
    df = df.drop_duplicates(subset=["date", "description", "amount"])
    removed = before - len(df)
    if removed:
        print(f"  {removed} duplicate rows removed.")

    # ── Step 9: Sort ──────────────────────────────────────────
    df = df.sort_values("date").reset_index(drop=True)

    date_range = (f"{df['date'].min().strftime('%d %b %Y')} "
                  f"to {df['date'].max().strftime('%d %b %Y')}")
    print(f"  Period:   {date_range}")
    print(f"  Income:   {len(df[df['amount'] > 0])} transactions")
    print(f"  Expenses: {len(df[df['amount'] < 0])} transactions")

    return df


def categorise_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Auto-assigns categories based on keyword matching.
    Preserves existing categories if already set.
    """
    df = df.copy()

    for idx, row in df.iterrows():
        if row["category"] not in ["Uncategorised", "", None]:
            continue
        desc  = str(row["description"]).lower()
        found = False
        for category, keywords in KEYWORD_MAP.items():
            if any(kw in desc for kw in keywords):
                df.at[idx, "category"] = category
                found = True
                break
        if not found:
            # income = positive amount
            if row["amount"] > 0:
                df.at[idx, "category"] = "Income"
            else:
                df.at[idx, "category"] = "Other"

    categorised = (df["category"] != "Uncategorised").sum()
    print(f"Categorisation complete. {categorised}/{len(df)} transactions assigned.")
    return df


def flag_essential_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Auto-flags known essential merchants.
    User can override in the edit step.
    """
    df = df.copy()

    flagged_count = 0
    for idx, row in df.iterrows():
        desc = str(row["description"]).lower()
        if any(kw in desc for kw in ESSENTIAL_KEYWORDS):
            df.at[idx, "essential"] = True
            flagged_count += 1

    print(f"Essential flagging complete. {flagged_count} transactions flagged.")
    return df


# ── Quick test ────────────────────────────────────────────────
print("\nCell 4 ready — data loading and categorisation functions defined.")


Cell 4 ready — data loading and categorisation functions defined.


In [87]:
# Test your function

from google.colab import files

uploaded = files.upload()

# automatically get the filename regardless of what it's called
filename = list(uploaded.keys())[0]
print(f"Uploaded: {filename}")

clean_data = load_and_clean_transaction_data(filename)
print(clean_data)

Saving transactions_3months.csv to transactions_3months (1).csv
Uploaded: transactions_3months (1).csv
Loaded 134 rows.
  Period:   03 Jan 2025 to 31 Mar 2025
  Income:   13 transactions
  Expenses: 121 transactions
          date                  description  amount           category  \
0   2025-01-03   CASUAL WORK - HUNGRY JACKS   580.0             Income   
1   2025-01-03     RENT - REALMARK PROPERTY -1050.0  Bills & Utilities   
2   2025-01-04  COLES SUPERMARKET KARRINYUP   -72.3   Food & Groceries   
3   2025-01-05     UBER EATS - HUNGRY JACKS   -17.5         Eating Out   
4   2025-01-06           BOOST JUICE HAY ST    -8.5    Coffee & Snacks   
..         ...                          ...     ...                ...   
129 2025-03-28    THE RECHABITE NORTHBRIDGE   -71.0             Social   
130 2025-03-28   UBER - NORTHBRIDGE TO HOME   -22.3          Transport   
131 2025-03-29  COLES SUPERMARKET KARRINYUP   -63.1   Food & Groceries   
132 2025-03-30         7-ELEVEN BEAUFORT ST 

In [88]:
# ── Cell 5: Spending Analysis Engine ─────────────────────────

def analyse_spending_by_category(df: pd.DataFrame) -> dict:
    """
    Analyses cleaned transaction data by category.
    Calculates totals, percentages, and generates student-focused insights.

    Parameters:
        df: cleaned DataFrame from load_and_clean_transaction_data()
    Returns:
        Dictionary containing full analysis summary
    """

    if df is None or df.empty:
        print("No transaction data found. Please load your CSV first.")
        return None

    # ── Step 1: Separate transaction types ───────────────────
    income_df        = df[df["amount"] > 0].copy()
    essential_df     = df[(df["essential"] == True) & (df["amount"] < 0)].copy()
    discretionary_df = df[(df["essential"] == False) & (df["amount"] < 0)].copy()
    refunds_df       = df[(df["amount"] > 0) &
                          (~df["category"].isin(["Income"]))].copy()

    # ── Step 2: Top level totals ──────────────────────────────
    total_income        = income_df["amount"].sum()
    total_essential     = abs(essential_df["amount"].sum())
    total_refunds       = refunds_df["amount"].sum()
    gross_discretionary = abs(discretionary_df["amount"].sum())
    net_discretionary   = gross_discretionary - total_refunds
    net_position        = total_income - total_essential - net_discretionary
    total_expenses      = total_essential + net_discretionary
    essential_pct       = (total_essential / total_income * 100) if total_income > 0 else 0
    savings_rate        = (net_position / total_income * 100) if total_income > 0 else 0

    # ── Step 3: Category breakdown (all expenses) ────────────
    all_expenses_df  = df[df["amount"] < 0].copy()
    category_summary = (
        all_expenses_df
        .groupby("category")["amount"]
        .agg(["sum", "count", "mean"])
        .rename(columns={"sum": "total", "count": "transactions", "mean": "average"})
    )
    category_summary["total"]   = category_summary["total"].abs()
    category_summary["average"] = category_summary["average"].abs()

    total_all_expenses = category_summary["total"].sum()
    category_summary["percentage"] = (
        category_summary["total"] / total_all_expenses * 100
    ).round(1)

    essential_categories = (
        all_expenses_df[all_expenses_df["essential"] == True]["category"]
        .unique().tolist()
    )
    category_summary["essential"] = category_summary.index.isin(essential_categories)
    category_summary = category_summary.sort_values("total", ascending=False)

    # ── Step 4: Invisible spending ────────────────────────────
    invisible_categories = ["Coffee & Snacks", "Eating Out", "Other"]
    invisible_df    = discretionary_df[
        (discretionary_df["amount"].abs() < 15) &
        (discretionary_df["category"].isin(invisible_categories))
    ]
    invisible_total  = abs(invisible_df["amount"].sum())
    invisible_count  = len(invisible_df)
    invisible_annual = invisible_total * 12

    # ── Step 5: Subscription audit ───────────────────────────
    subscription_df     = discretionary_df[
        discretionary_df["category"] == "Subscriptions"
    ].copy()
    subscription_total  = abs(subscription_df["amount"].sum())
    subscription_annual = subscription_total * 12

    # ── Step 6: Social spending ───────────────────────────────
    social_total = abs(
        discretionary_df[
            discretionary_df["category"] == "Social"]["amount"].sum()
    )
    social_pct = (social_total / net_discretionary * 100) if net_discretionary > 0 else 0

    if social_pct < 5:
        social_label = "Very low — social connection matters for wellbeing"
    elif social_pct <= 20:
        social_label = "Reasonable"
    elif social_pct <= 35:
        social_label = "High — worth reviewing"
    else:
        social_label = "Excessive — this is significantly impacting your budget"

    # ── Step 7: Day of week pattern ───────────────────────────
    discretionary_df = discretionary_df.copy()
    discretionary_df["day_of_week"] = discretionary_df["date"].dt.day_name()
    day_spend = (
        discretionary_df.groupby("day_of_week")["amount"]
        .sum().abs()
        .reindex(["Monday","Tuesday","Wednesday","Thursday",
                  "Friday","Saturday","Sunday"])
        .fillna(0)
    )
    highest_spend_day = day_spend.idxmax()

    # ── Step 8: Financial health score ───────────────────────
    score = 100

    # essential expenses as % of income
    if essential_pct > 60:
        score -= 20
    elif essential_pct > 50:
        score -= 10

    # budget adherence
    if net_position < 0:
        overspend_pct = abs(net_position) / total_income * 100
        score -= min(30, int(overspend_pct))

    # invisible spending
    if invisible_total > 0:
        invisible_pct = invisible_total / net_discretionary * 100
        if invisible_pct > 15:
            score -= 10
        elif invisible_pct > 8:
            score -= 5

    # subscriptions
    if subscription_total > 0:
        sub_pct = subscription_total / net_discretionary * 100
        if sub_pct > 20:
            score -= 10
        elif sub_pct > 10:
            score -= 5

    # savings rate
    if savings_rate < 0:
        score -= 20
    elif savings_rate < 10:
        score -= 10

    # social overspending
    if social_pct > 35:
        score -= 10
    elif social_pct > 25:
        score -= 5

    score = max(0, min(100, score))

    if score >= 80:
        health_label, health_colour = "On Track",        "green"
    elif score >= 60:
        health_label, health_colour = "Needs Attention", "orange"
    elif score >= 40:
        health_label, health_colour = "At Risk",         "red"
    else:
        health_label, health_colour = "Action Required", "darkred"

    # ── Step 9: Generate insights ─────────────────────────────
    insights = []

    if net_position < 0:
        insights.append({
            "severity": "critical",
            "title":    "Spending Exceeds Income",
            "detail":   (
                f"You spent ${abs(net_position):.2f} more than you earned. "
                f"This is not sustainable. Essential expenses alone account for "
                f"{essential_pct:.1f}% of your income (${total_essential:.2f}), "
                f"leaving only ${total_income - total_essential:.2f} for everything else. "
                f"Your discretionary spending of ${net_discretionary:.2f} exceeds that by "
                f"${net_discretionary - (total_income - total_essential):.2f}."
            )
        })
    elif essential_pct > 50:
        insights.append({
            "severity": "warning",
            "title":    "High Fixed Cost Burden",
            "detail":   (
                f"Essential expenses take up {essential_pct:.1f}% of your income. "
                f"This is a significant constraint. You have only "
                f"${total_income - total_essential:.2f} per period for all "
                f"discretionary spending and savings combined."
            )
        })

    if invisible_total > 0:
        insights.append({
            "severity": "warning",
            "title":    "Invisible Spending Detected",
            "detail":   (
                f"${invisible_total:.2f} was spent across {invisible_count} small "
                f"transactions under $15. That is ${invisible_annual:.2f} per year "
                f"on purchases individually too small to notice but collectively "
                f"significant. Coffees, snacks, and convenience store visits "
                f"are the usual culprits."
            )
        })

    if subscription_total > 0:
        sub_pct = subscription_total / net_discretionary * 100
        insights.append({
            "severity": "warning" if sub_pct > 10 else "info",
            "title":    "Subscription Costs",
            "detail":   (
                f"Subscriptions cost ${subscription_total:.2f} per month "
                f"(${subscription_annual:.2f} per year) — {sub_pct:.1f}% of "
                f"discretionary spending. Review each one: if you have not used "
                f"a service at least 4 times this month, cancel it."
            )
        })

    if social_pct > 25:
        insights.append({
            "severity": "warning",
            "title":    "Social Spending is High",
            "detail":   (
                f"Social spending of ${social_total:.2f} represents {social_pct:.1f}% "
                f"of your discretionary budget. Some social spending is healthy and "
                f"necessary, but at this level it is a primary driver of budget pressure. "
                f"Setting a fixed weekly social limit and sticking to it would have "
                f"an immediate impact."
            )
        })

    # top discretionary category
    disc_summary = category_summary[category_summary["essential"] == False]
    if not disc_summary.empty:
        top_cat      = disc_summary.index[0]
        top_cat_row  = disc_summary.iloc[0]
        if top_cat not in ["Social"] or social_pct <= 25:
            insights.append({
                "severity": "info",
                "title":    f"Largest Discretionary Category: {top_cat}",
                "detail":   (
                    f"{top_cat} accounts for ${top_cat_row['total']:.2f} "
                    f"({top_cat_row['percentage']:.1f}% of all expenses). "
                    f"Reducing this category by 25% would save "
                    f"${top_cat_row['total'] * 0.25:.2f} per period."
                )
            })

    insights.append({
        "severity": "info",
        "title":    f"Highest Spending Day: {highest_spend_day}",
        "detail":   (
            f"You tend to spend the most on {highest_spend_day}s. "
            f"Being aware of this pattern can help you pause before "
            f"discretionary purchases on that day."
        )
    })

    # ── Step 10: Print console summary ───────────────────────
    print("=" * 55)
    print("         SMART FINANCE ASSISTANT — SUMMARY")
    print("=" * 55)
    print(f"  Total Income:          ${total_income:>9.2f}")
    print(f"  Essential Expenses:    ${total_essential:>9.2f}  ({essential_pct:.1f}% of income)")
    print(f"  Discretionary Spend:   ${net_discretionary:>9.2f}")
    if net_position < 0:
        print(f"  Net Position:          ${net_position:>9.2f}  ** DEFICIT **")
    else:
        print(f"  Net Position:          ${net_position:>9.2f}")
    print(f"  Health Score:          {score}/100 — {health_label}")
    print(f"{'─' * 55}")
    print(f"  {'Category':<22} {'Total':>8}  {'%':>5}  {'Type':<12}")
    print(f"{'─' * 55}")
    for cat, row in category_summary.iterrows():
        cat_type = "Essential" if row["essential"] else "Flexible"
        print(f"  {cat:<22} ${row['total']:>7.2f}  {row['percentage']:>4.1f}%  {cat_type}")
    print("=" * 55)

    return {
        "total_income":        total_income,
        "total_essential":     total_essential,
        "net_discretionary":   net_discretionary,
        "net_position":        net_position,
        "total_expenses":      total_expenses,
        "essential_pct":       essential_pct,
        "savings_rate":        savings_rate,
        "category_summary":    category_summary,
        "invisible_total":     invisible_total,
        "invisible_count":     invisible_count,
        "invisible_annual":    invisible_annual,
        "subscription_total":  subscription_total,
        "subscription_annual": subscription_annual,
        "subscription_df":     subscription_df,
        "social_total":        social_total,
        "social_pct":          social_pct,
        "social_label":        social_label,
        "day_spend":           day_spend,
        "highest_spend_day":   highest_spend_day,
        "health_score":        score,
        "health_label":        health_label,
        "health_colour":       health_colour,
        "insights":            insights,
    }


print("Cell 5 ready — analysis engine defined.")

Cell 5 ready — analysis engine defined.


In [18]:
# Flag Essential Spending
def flag_essential_transactions(df):
    """
    Auto-flags known essential merchants as essential.
    """
    essential_keywords = [
        'rent', 'synergy', 'alinta', 'insurance', 'realmark',
        'ray white', 'lj hooker', 'optus', 'telstra', 'medicare',
        'electricity', 'water', 'internet', 'gas'
    ]

    for keyword in essential_keywords:
        mask = df['description'].str.lower().str.contains(keyword, na=False)
        df.loc[mask, 'essential'] = True

    flagged = df['essential'].sum()
    print(f"✅ {flagged} transactions automatically flagged as essential.")
    print("\n🔒 Essential transactions:")
    print(df[df['essential'] == True][['date', 'description', 'amount']].to_string(index=False))
    print("\n  ℹ️  You can manually update the essential column if anything looks wrong.")

    return df

In [19]:
# Test your function

# Step 1 - flag essentials first
clean_data = flag_essential_transactions(clean_data)

# Step 2 - run analysis (no print wrapping it)
analysis = analyse_spending_by_category(clean_data)

✅ 3 transactions automatically flagged as essential.

🔒 Essential transactions:
      date              description  amount
2024-03-08 RENT - REALMARK PROPERTY  -950.0
2024-03-09       SYNERGY POWER BILL  -143.2
2024-03-22         OPTUS PHONE BILL   -45.0

  ℹ️  You can manually update the essential column if anything looks wrong.
        💰 SMART FINANCE ASSISTANT SUMMARY

📥  Total Income:            $ 1421.99
🔒  Essential Expenses:      $ 1138.20
💸  Discretionary Spending:  $ 1009.79
💚  Net Position:            $ -726.00

  ℹ️  Essential expenses are excluded from budget analysis.
  Set your budget in the next step based on these figures.

───────────────────────────────────────────────────────
📂  SPENDING BY CATEGORY
───────────────────────────────────────────────────────
Category                  Total   Txns      Avg      %
───────────────────────────────────────────────────────
Shopping               $ 538.98      3  $179.66  53.4%
Food & Groceries       $ 196.33      3  $ 65.44  

In [23]:
# 🤖 AI Collaboration: Business Insights Generator

def generate_ai_recommendations(analysis_data):
    """
    Sends spending analysis to the AI and returns personalised
    financial recommendations formatted for university students.

    Args:
        analysis_data: Dictionary with spending analysis results
    Returns:
        str: Formatted recommendations report
    """

    if analysis_data is None:
        print("❌ No analysis data found. Please run analyse_spending_by_category() first.")
        return None

    # ── Step 1: Extract key figures from analysis ─────────────
    total_income        = analysis_data['total_income']
    total_essential     = analysis_data['total_essential']
    net_discretionary   = analysis_data['net_discretionary']
    net_position        = analysis_data['net_position']
    invisible_total     = analysis_data['invisible_total']
    invisible_count     = analysis_data['invisible_count']
    subscription_total  = analysis_data['subscription_total']
    subscription_annual = analysis_data['subscription_annual']
    social_total        = analysis_data['social_total']
    social_percentage   = analysis_data['social_percentage']
    social_label        = analysis_data['social_label']
    category_summary    = analysis_data['category_summary']

    # ── Step 2: Build category breakdown string for AI ────────
    category_lines = []
    for category, row in category_summary.iterrows():
        category_lines.append(
            f"  - {category}: ${row['total']:.2f} "
            f"({row['percentage']}% of discretionary spend, "
            f"{int(row['transactions'])} transactions, "
            f"avg ${row['average']:.2f} each)"
        )
    category_text = "\n".join(category_lines)

    # ── Step 3: Build full context prompt for AI ──────────────
    prompt = f"""
You are a friendly, encouraging financial coach for Australian university students.
Analyse the following spending data and provide personalised recommendations.

STUDENT FINANCIAL SUMMARY:
- Total Income this period:       ${total_income:.2f}
- Essential Expenses (rent etc):  ${total_essential:.2f}
- Discretionary Spending:         ${net_discretionary:.2f}
- Net Position:                   ${net_position:.2f}

SPENDING BREAKDOWN BY CATEGORY:
{category_text}

ADDITIONAL INSIGHTS:
- Invisible spending (transactions under $15): ${invisible_total:.2f} across {invisible_count} transactions
- Subscriptions total: ${subscription_total:.2f}/month (${subscription_annual:.2f}/year)
- Social spending: ${social_total:.2f} ({social_percentage:.1f}% of discretionary) — {social_label}

Please provide a financial recommendations report with the following sections:

1. SPENDING SNAPSHOT — a brief 2-3 sentence summary of the student's overall financial picture, be honest but encouraging

2. TOP SAVINGS OPPORTUNITIES — identify the top 3 specific areas where money can be saved, with a dollar amount saved per month if the advice is followed. Never target essential expenses like rent or utilities.

3. INVISIBLE SPENDING ALERT — comment specifically on the small frequent purchases and what they add up to over a year

4. SUBSCRIPTION AUDIT — review the subscription spending and suggest whether any should be reconsidered

5. SOCIAL SPENDING — acknowledge social spending positively if healthy, or gently flag if excessive. Remind the student that social connection is important and should be budgeted for, not eliminated.

6. ONE SMALL WIN — suggest one single easy change the student can make this week that would have an immediate impact

Keep the tone friendly, specific, and non-judgmental. Use Australian context where relevant.
Avoid generic advice — every recommendation should reference the student's actual numbers.
Do not suggest cutting essential expenses like rent, power, or phone bills.
"""

    # ── Step 4: Send to AI and get response ───────────────────
    print("🤖 Generating personalised recommendations...")
    print("─" * 55)

    try:
        response = get_response(prompt)
    except Exception as e:
        print(f"❌ Could not reach AI: {e}")
        print("  Check your hands-on-ai server connection and try again.")
        return None

    # ── Step 5: Display formatted report ─────────────────────
    print("\n" + "=" * 55)
    print("     🎯 YOUR PERSONALISED FINANCIAL RECOMMENDATIONS")
    print("=" * 55)
    print(response)
    print("=" * 55)

    return response

# Test your function
# recommendations = generate_financial_recommendations(analysis)
# print(recommendations)

In [24]:
#Test AI Recommendations
ai_recommendations = generate_ai_recommendations(analysis)

🤖 Generating personalised recommendations...
───────────────────────────────────────────────────────

     🎯 YOUR PERSONALISED FINANCIAL RECOMMENDATIONS
Okay, let’s get your finances sorted out! Here’s a personalised report based on your spending data – let’s tackle this together and get you feeling more in control.

**1. SPENDING SNAPSHOT**

Right now, things are a little tight, but it’s fantastic that you’re taking a look at your spending. You've got a decent income, but it’s currently being eaten up by discretionary spending, leaving you with a negative net position. Don’t worry, this is a common situation for uni students – small adjustments can make a massive difference and get you back on track. 

**2. TOP SAVINGS OPPORTUNITIES**

Let’s focus on some specific areas for improvement. Here’s how we can make a real impact this month:

*   **Shopping:** Reducing your shopping spend by just $200 per month (about 40% of the category) would significantly shift things. You're spending a f

---

# 🌐 Advanced Features: Integrating AI Components

Once your foundation data processing is solid, integrate advanced AI features using hands-on-ai.

## Chat Interface Integration

In [33]:
from hands_on_ai.chat import get_response

def create_finance_chat_personality(analysis_data=None):
    """
    Sets up a finance-focused chat personality with full awareness
    of the student's actual spending data.

    Args:
        analysis_data: Optional dictionary from analyse_spending_by_category()
    Returns:
        tuple: system_prompt string and conversation history list
    """

    # ── Step 1: Build spending context if analysis is available ──
    if analysis_data is not None:
        spending_context = f"""
You have access to this student's actual financial data:
- Total Income:             ${analysis_data['total_income']:.2f}
- Essential Expenses:       ${analysis_data['total_essential']:.2f}
- Discretionary Spending:   ${analysis_data['net_discretionary']:.2f}
- Net Position:             ${analysis_data['net_position']:.2f}
- Invisible Spending Total: ${analysis_data['invisible_total']:.2f}
- Subscription Cost:        ${analysis_data['subscription_total']:.2f}/month
- Social Spending:          ${analysis_data['social_total']:.2f} — {analysis_data['social_label']}

Top spending categories:
"""
        for category, row in analysis_data['category_summary'].iterrows():
            spending_context += (
                f"  - {category}: ${row['total']:.2f} "
                f"({row['percentage']}%)\n"
            )
        spending_context += """
Always reference these real numbers when giving advice.
Never give generic advice when you have actual data to work with.
"""
    else:
        spending_context = """
No spending data has been loaded yet.
If the student asks about their specific spending, encourage them
to upload their CSV file first for personalised advice.
"""

    # ── Step 2: Build the system personality prompt ───────────
    system_prompt = f"""
You are Finn, a friendly and practical financial coach built specifically
for Australian university students. You are warm, encouraging, and never
judgmental about past spending mistakes — you focus on progress, not perfection.

YOUR PERSONALITY:
- Friendly and conversational, like a smart older friend who knows finance
- Use Australian context naturally (Centrelink, Transperth, Woolies, etc.)
- Keep responses concise — students want quick wins, not lectures
- Celebrate small wins enthusiastically
- Never make the student feel guilty about social spending
- Use relatable examples (coffee runs, Uber Eats, share house bills)

YOUR RULES:
- Never suggest cutting essential expenses like rent, utilities, or phone bills
- Always protect a reasonable social budget
- When giving savings advice, state the dollar impact per month and per year
- Keep responses to 4-6 sentences unless a detailed breakdown is asked for
- If asked something outside personal finance, redirect back to finance topics

{spending_context}
"""

    conversation_history = []

    print("=" * 55)
    print("        💬 FINN — YOUR FINANCE COACH")
    print("=" * 55)
    print("\n  Hey! I'm Finn, your personal finance coach 👋")
    print("  I've looked at your spending data and I'm ready to help.")
    print("  Ask me anything about your budget or savings goals.")
    print("\n" + "─" * 55)
    print("  Type 'quit' to exit the chat.")
    print("─" * 55 + "\n")

    return system_prompt, conversation_history

In [35]:
def run_finance_chat(analysis_data=None):
    """
    Runs an interactive chat session with Finn the finance coach.
    """
    system_prompt, history = create_finance_chat_personality(analysis_data)

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("\nFinn: Good chat! Small changes add up fast. See you next time 👋")
            break

        if not user_input:
            continue

        # build full conversation context for AI
        conversation_context = system_prompt + "\n\nConversation so far:\n"
        for message in history:
            role = "Student" if message['role'] == "user" else "Finn"
            conversation_context += f"{role}: {message['content']}\n"
        conversation_context += f"\nStudent: {user_input}\nFinn:"

        # get response
        try:
            response = get_response(conversation_context)
        except Exception as e:
            print(f"❌ Could not reach AI: {e}")
            break

        # store in history
        history.append({"role": "user",    "content": user_input})
        history.append({"role": "assistant", "content": response})

        print(f"\nFinn: {response}\n")

In [36]:
# Test get_response is working
test_response = get_response("In one sentence, what is the most important money habit for a university student?")
print("✅ get_response is working!")
print(f"\nFinn says: {test_response}")

✅ get_response is working!

Finn says: The most important money habit for a university student is consistently tracking their spending to understand where their money is going and make informed decisions about budgeting.


## RAG System for Financial Documents

In [89]:
# ── Cell 6: RAG Knowledge Base ────────────────────────────────

from hands_on_ai import rag
from pathlib import Path

FINANCE_KNOWLEDGE = """
BUDGETING BASICS
The 50/30/20 rule is a starting framework: 50% of income to needs, 30% to wants, and 20% to savings. For students on irregular income, cover essentials first, set a fixed weekly discretionary limit, and automate savings transfers on payday. Tracking every transaction, even small ones, is the single most effective budgeting habit. A budget only works if it reflects reality — set limits based on what you actually spend, then reduce gradually.

COFFEE AND SNACKS
Coffee and snack purchases are the classic invisible spending trap. A $5 coffee bought 5 days a week costs $1,300 per year. Swapping 3 of those for home brews weekly saves around $780 annually. The issue is not one coffee — it is the habit of daily convenience spending that accumulates without notice.

SUBSCRIPTIONS
Subscription creep is one of the most common budget leaks for students. List every subscription and ask: did I use this at least 4 times last month? Sharing plans with housemates halves costs immediately. A $15 subscription unused for 6 months has cost $90 for nothing. Streaming services, gym memberships, and app subscriptions should be audited every 3 months without exception.

FOOD AND GROCERIES
Switching from Woolworths or Coles to ALDI for staples saves 20 to 30 percent on average. Meal prepping on Sundays eliminates the temptation to order delivery during busy uni weeks. Writing a shopping list and sticking to it prevents impulse purchases. Buying in bulk for non-perishables reduces cost per unit significantly over time.

EATING OUT AND FOOD DELIVERY
Food delivery apps are the single biggest discretionary budget threat for students. A $25 Uber Eats order twice a week totals $2,600 per year. The convenience premium on delivery is typically 30 to 50 percent above cooking at home. Setting a hard limit of two takeaway meals per week and cooking the rest saves $100 to $150 per month without significant lifestyle sacrifice.

SOCIAL SPENDING
Social spending should be budgeted for, not eliminated. Cutting social activities entirely leads to burnout and the abandonment of budgets altogether. Set a fixed social budget per week and treat it as a real limit. Pre-drinks at home, free campus events, and cheaper venue choices still allow a full social life at a fraction of the cost. The goal is controlled social spending, not zero social spending.

EXCESSIVE SOCIAL SPENDING
When social spending exceeds 25 percent of discretionary income it becomes a primary budget problem. Alcohol, Uber rides home, entry fees, and late-night food combine quickly. A single big night out can cost $100 to $150 all-in. Setting a per-night cash limit and leaving the card at home is one of the most effective controls. Choosing venues with no entry fee, drinking less, and splitting Ubers all reduce the per-outing cost significantly.

INVISIBLE SPENDING
Invisible spending is money that leaves the account in amounts too small to notice individually but large enough to matter in total. Convenience store visits, vending machines, small app purchases, and ATM fees are common culprits. Most students find $80 to $150 disappearing this way each month when they total it up. The fix is simple: categorise and total all transactions under $15 once a month and confront the number directly.

SHOPPING AND IMPULSE BUYING
The 48-hour rule is highly effective for non-essential purchases over $50 — wait two days before buying. Unsubscribing from retail email lists removes the trigger. Selling unused items on Facebook Marketplace before buying new things creates a natural friction. Most impulse purchases feel less urgent 48 hours later.

SAVINGS GOALS
Saving is significantly easier when tied to a specific goal with a deadline. Name the goal, set the amount, set the date, and work backwards to a weekly savings target. Automating a transfer to a separate savings account on payday removes the decision entirely. Even $20 per week builds a $1,000 emergency fund in less than a year.

SAFETY NET FUND
A safety net fund is more important than a holiday savings goal. Casual workers have no sick leave — one bad week of illness can mean missing rent. A $500 buffer covers one missed pay. A $1,000 buffer covers two weeks of illness or a car repair. One month of essential expenses as a buffer covers most realistic emergencies. Build the safety net before saving for anything else.

EMERGENCY FUND
Without an emergency fund, any unexpected cost goes on a credit card or wipes out a savings goal. Start with $500, then build to one month of essential expenses. Keep it in a separate account so it is out of sight. Do not touch it for non-emergencies — it is not a holiday fund.

PAYDAY HABITS
Payday splurging is extremely common. The feeling of a full account triggers spending that unravels the entire month's budget within days. Automate transfers on payday in this order: savings first, then bills, then a weekly allowance. Treat the weekly allowance as the only available money, even if more is saved elsewhere.

TRANSPORT
A Transperth concession SmartRider is significantly cheaper than driving and parking in Perth. Uber rides home from nights out add up fast — splitting with friends or pre-booking reduces the cost. Carpooling to uni cuts petrol and parking costs by half. These small transport choices compound over a semester.

INCOME AND IRREGULAR WORK
Base the budget on minimum expected income, not the best week. Casual work income is unreliable — one roster change can cut weekly earnings significantly. Centrelink Youth Allowance and Austudy provide a reliable baseline. Any income above the minimum should go to savings, not lifestyle inflation. Never increase fixed commitments based on irregular income.

ENTERTAINMENT OVERSPENDING
Set a fixed entertainment budget and treat it as a hard limit. When it runs out for the week, stop. Free alternatives include campus events, parks, libraries, and streaming shared with housemates. The problem is not entertainment itself — it is entertainment without a limit.

RENT PRESSURE
When rent exceeds 30 percent of income, financial pressure is significant. At 40 percent or above, there is mathematically very little room for anything else. Rent cannot typically be reduced in the short term but it should be a primary consideration when renewing leases or choosing accommodation. Living closer to university or with more housemates directly reduces this pressure.
"""

def setup_finance_rag():
    """
    Builds the RAG knowledge base from the finance knowledge text.
    Uses keyword-based retrieval since embeddings are not supported
    on the current server.

    Returns:
        callable: ask() function
    """

    # save knowledge to file and chunk it
    knowledge_path = Path("finance_knowledge.txt")
    with open(knowledge_path, "w") as f:
        f.write(FINANCE_KNOWLEDGE)

    text   = rag.load_text_file(knowledge_path)
    chunks = rag.chunk_text(text)
    print(f"Knowledge base ready. {len(chunks)} chunks loaded.")

    def ask(question: str, k: int = 3) -> str:
        """
        Retrieves the most relevant knowledge chunks for a question
        and generates a grounded AI answer.

        Args:
            question: finance question to answer
            k:        number of chunks to retrieve
        Returns:
            str: grounded AI response
        """

        # keyword scoring
        question_words = set(question.lower().split())
        scored = sorted(
            [(len(question_words & set(c.lower().split())), c) for c in chunks],
            key=lambda x: x[0],
            reverse=True
        )
        context = "\n\n".join(chunk for _, chunk in scored[:k])

        prompt = f"""
You are a direct, no-nonsense financial advisor for Australian university students.
Answer using ONLY the retrieved knowledge below.
Be specific, reference dollar amounts where possible, and do not soften bad news.
Keep your answer to 4 sentences maximum.

KNOWLEDGE:
{context}

QUESTION: {question}

ANSWER:"""

        try:
            return get_response(prompt)
        except Exception as e:
            return f"Could not reach AI: {e}"

    return ask


# initialise RAG
finance_rag_ask = setup_finance_rag()


# wrapper object for consistent syntax
class FinanceRAG:
    def ask(self, question):
        return finance_rag_ask(question)

rag_system = FinanceRAG()

print("Cell 6 ready — RAG system initialised.")

Knowledge base ready. 3 chunks loaded.
Cell 6 ready — RAG system initialised.


In [45]:
# 🤖 AI Collaboration: Document Retrieval Setup

def setup_financial_rag(chunks):
    """
    Sets up keyword-based RAG using pre-chunked finance knowledge.
    Retrieves relevant chunks by keyword matching, then generates
    a grounded AI answer using get_response().

    Args:
        chunks: list of text chunks from rag.chunk_text()
    Returns:
        callable: ask() function ready to use
    """

    def ask(question, k=3):
        """
        Retrieves the most relevant knowledge chunks for a question
        and generates a grounded AI answer.

        Args:
            question: student's finance question
            k:        number of chunks to retrieve
        Returns:
            str: AI answer grounded in retrieved knowledge
        """

        # ── Step 1: Score each chunk by keyword matches ───────
        question_words = set(question.lower().split())
        scored_chunks  = []

        for chunk in chunks:
            chunk_words = set(chunk.lower().split())
            score       = len(question_words & chunk_words)
            scored_chunks.append((score, chunk))

        # ── Step 2: Sort and take top k chunks ────────────────
        scored_chunks.sort(key=lambda x: x[0], reverse=True)
        top_chunks = [chunk for _, chunk in scored_chunks[:k]]
        context    = "\n\n".join(top_chunks)

        # ── Step 3: Build grounded prompt ─────────────────────
        prompt = f"""
You are Finn, a friendly financial coach for Australian university students.
Use ONLY the retrieved knowledge below to answer the question.
Keep your answer to 3-5 sentences, friendly and specific.
Never suggest cutting essential expenses like rent or utilities.
Always protect a reasonable social budget.

RETRIEVED KNOWLEDGE:
{context}

STUDENT QUESTION:
{question}

FINN'S ANSWER:
"""

        # ── Step 4: Generate and return answer ────────────────
        try:
            response = get_response(prompt)
            return response
        except Exception as e:
            return f"❌ Could not reach AI: {e}"

    print("✅ RAG system ready — using keyword retrieval")
    return ask


# Initialise the RAG ask function
finance_rag_ask = setup_financial_rag(chunks)


✅ RAG system ready — using keyword retrieval


In [48]:
# Wrap in a simple object so rag.ask() syntax works
class RAG:
    def ask(self, question):
        return finance_rag_ask(question)

rag_system = RAG()

In [50]:
#Test RAG System
answer = rag_system.ask("What's a good budgeting strategy for someone who overspends on entertainment?")
print(answer)

Hey there! It’s great you’re being proactive about your spending – that’s half the battle! For someone who tends to overspend on entertainment, I’d suggest setting a fixed weekly budget – let’s say $30 – and sticking to it like glue. Tracking every entertainment expense, even small things like a movie ticket, will really highlight where you're going over. Also, remember that free or cheap alternatives like campus events, parks, or streaming with friends can still give you a great social life without breaking the bank – try to use those first!


## Custom Financial Tools

In [92]:
# ── Cell 7: Agent Tool — Savings Calculator ───────────────────

from hands_on_ai import agent

def create_savings_calculator_tool():
    """
    Registers a savings goal calculator as an agent tool.
    Accepts a single string input for hands_on_ai agent compatibility.
    """

    def savings_goal_calculator(input: str) -> str:
        """
        Calculates time to reach a savings goal.
        Input format: "target=2000, monthly=150, current=350"
        """

        # ── Parse input string ────────────────────────────────
        try:
            params = {}
            for part in input.split(","):
                if "=" in part:
                    key, value = part.strip().split("=")
                    params[key.strip()] = float(value.strip())

            target_amount        = params.get("target", 0)
            monthly_contribution = params.get("monthly", 0)
            current_savings      = params.get("current", 0)
        except Exception:
            return (
                "Could not read inputs. "
                "Please provide: target=2000, monthly=150, current=350"
            )

        # ── Validate ──────────────────────────────────────────
        if target_amount <= 0:
            return "Target amount must be greater than $0."
        if monthly_contribution <= 0:
            return "Monthly contribution must be greater than $0."
        if current_savings < 0:
            return "Current savings cannot be negative."
        if current_savings >= target_amount:
            return (
                f"Goal already reached. "
                f"Current savings ${current_savings:.2f} meets "
                f"target of ${target_amount:.2f}."
            )

        # ── Calculate ─────────────────────────────────────────
        amount_remaining    = target_amount - current_savings
        months_to_goal      = amount_remaining / monthly_contribution
        full_months         = int(months_to_goal)
        extra_weeks         = round((months_to_goal - full_months) * 4)
        years               = full_months // 12
        months              = full_months % 12
        weekly_equivalent   = monthly_contribution / 4.33
        progress_pct        = (current_savings / target_amount) * 100

        # ── Build time string ─────────────────────────────────
        if years > 0 and months > 0:
            time_str = (f"{years} year{'s' if years > 1 else ''} "
                        f"and {months} month{'s' if months > 1 else ''}")
        elif years > 0:
            time_str = f"{years} year{'s' if years > 1 else ''}"
        elif full_months > 0 and extra_weeks > 0:
            time_str = (f"{full_months} month{'s' if full_months > 1 else ''} "
                        f"and {extra_weeks} week{'s' if extra_weeks > 1 else ''}")
        elif full_months > 0:
            time_str = f"{full_months} month{'s' if full_months > 1 else ''}"
        else:
            time_str = f"{extra_weeks} week{'s' if extra_weeks > 1 else ''}"

        # ── Format output ─────────────────────────────────────
        return (
            f"SAVINGS GOAL CALCULATOR\n"
            f"{'─' * 38}\n"
            f"Goal:                  ${target_amount:>10.2f}\n"
            f"Already saved:         ${current_savings:>10.2f}\n"
            f"Still needed:          ${amount_remaining:>10.2f}\n"
            f"Progress:              {progress_pct:>9.1f}%\n"
            f"{'─' * 38}\n"
            f"Monthly contribution:  ${monthly_contribution:>10.2f}\n"
            f"Weekly equivalent:     ${weekly_equivalent:>10.2f}\n"
            f"Time to reach goal:    {time_str}\n"
            f"{'─' * 38}"
        )

    # ── Register with agent ───────────────────────────────────
    agent.register_tool(
        name        = "savings_goal_calculator",
        description = (
            "Calculates how long it will take to reach a savings goal. "
            "Input format: 'target=2000, monthly=150, current=350' "
            "where target is the goal amount, monthly is the monthly "
            "contribution, and current is the amount already saved."
        ),
        function    = savings_goal_calculator
    )

    print(f"Savings calculator registered.")
    print(f"  Registered tools: {agent.list_tools()}")
    return savings_goal_calculator


# initialise the tool
savings_calculator = create_savings_calculator_tool()


Savings calculator registered.
  Registered tools: [{'name': 'savings_goal_calculator', 'description': "Calculates how long it will take to reach a savings goal. Input format: 'target=2000, monthly=150, current=350' where target is the goal amount, monthly is the monthly contribution, and current is the amount already saved."}]


In [93]:
#Test Calucaltor
# Test the calculator directly using the string input format
print(savings_calculator("target=2000, monthly=150, current=350"))

SAVINGS GOAL CALCULATOR
──────────────────────────────────────
Goal:                  $   2000.00
Already saved:         $    350.00
Still needed:          $   1650.00
Progress:                   17.5%
──────────────────────────────────────
Monthly contribution:  $    150.00
Weekly equivalent:     $     34.64
Time to reach goal:    11 months
──────────────────────────────────────


In [64]:
#Test Calc 2
# Test via the agent so it uses the registered tool

result = agent.run_agent(
    "I want to save $2000 for a Bali trip. "
    "I have $350 saved already and can put away $150 per month. "
    "How long will it take?",
    verbose=True
)
print(result)

It will take 11 months to save $2000, given your current savings of $350 and a monthly contribution of $150.


## Gradio UI Integration

In [65]:
import gradio as gr
print(gr.__version__)

5.50.0


In [70]:
import gradio as gr

# ── Global state ──────────────────────────────────────────────
app_state = {
    "clean_data":   None,
    "analysis":     None,
    "chat_history": [],
    "system_prompt": None
}

def create_finance_assistant_ui():
    """
    Comprehensive Gradio interface for the Smart Finance Assistant.
    Four tabs: Upload & Edit, Spending Dashboard, Budget & Goals, AI Advice.
    """

    # ── Tab 1 handlers ────────────────────────────────────────
    def handle_csv_upload(file):
        if file is None:
            return "❌ No file uploaded.", None
        try:
            df = load_and_clean_transaction_data(file.name)
            df = flag_essential_transactions(df)
            app_state["clean_data"] = df
            app_state["analysis"]   = analyse_spending_by_category(df)
            return "✅ File loaded and analysed successfully!", df
        except Exception as e:
            return f"❌ Error loading file: {e}", None

    def handle_manual_entry(date, description, amount, category, essential):
        if app_state["clean_data"] is None:
            return "❌ Please upload a CSV file first.", None
        try:
            import pandas as pd
            new_row = pd.DataFrame([{
                "date":        pd.to_datetime(date, dayfirst=True),
                "description": description.upper(),
                "amount":      float(amount),
                "category":    category,
                "essential":   essential
            }])
            app_state["clean_data"] = pd.concat(
                [app_state["clean_data"], new_row], ignore_index=True
            )
            app_state["analysis"] = analyse_spending_by_category(
                app_state["clean_data"]
            )
            return "✅ Transaction added successfully!", app_state["clean_data"]
        except Exception as e:
            return f"❌ Error adding transaction: {e}", None

    def handle_table_edit(df):
        try:
            app_state["clean_data"] = df
            app_state["analysis"]   = analyse_spending_by_category(df)
            return "✅ Changes saved and analysis updated!"
        except Exception as e:
            return f"❌ Error saving changes: {e}"

    def handle_upload_and_display(file):
        status, df = handle_csv_upload(file)
        return status, df

    # ── Tab 2 handlers ────────────────────────────────────────
    def get_dashboard():
        if app_state["analysis"] is None:
            return "❌ No data loaded. Please upload a CSV file first."

        a = app_state["analysis"]
        lines = []
        lines.append("## 💰 Financial Overview")
        lines.append(f"- **Total Income:**           ${a['total_income']:.2f}")
        lines.append(f"- **Essential Expenses:**     ${a['total_essential']:.2f}")
        lines.append(f"- **Discretionary Spending:** ${a['net_discretionary']:.2f}")
        lines.append(f"- **Net Position:**           ${a['net_position']:.2f}")

        lines.append("\n## 📂 Spending by Category")
        lines.append("| Category | Total | Transactions | Average | % |")
        lines.append("|---|---|---|---|---|")
        for cat, row in a['category_summary'].iterrows():
            lines.append(
                f"| {cat} | ${row['total']:.2f} | "
                f"{int(row['transactions'])} | "
                f"${row['average']:.2f} | "
                f"{row['percentage']:.1f}% |"
            )

        lines.append("\n## 👻 Invisible Spending")
        lines.append(
            f"Small purchases under $15 total **${a['invisible_total']:.2f}** "
            f"across **{a['invisible_count']} transactions**."
        )

        lines.append("\n## 📋 Subscription Audit")
        lines.append(
            f"Subscriptions cost **${a['subscription_total']:.2f}/month** "
            f"(${a['subscription_annual']:.2f}/year)."
        )

        lines.append("\n## 🎉 Social Spending")
        lines.append(
            f"Social spending: **${a['social_total']:.2f}** "
            f"({a['social_percentage']:.1f}%) — {a['social_label']}"
        )

        lines.append("\n## 💡 Key Insights")
        for insight in a['insights']:
            lines.append(f"- {insight}")

        return "\n".join(lines)

    # ── Tab 3 handlers ────────────────────────────────────────
    def calculate_budget_plan(monthly_budget, goal_name, goal_amount, goal_date):
        if app_state["analysis"] is None:
            return "❌ No data loaded. Please upload a CSV file first."

        a = app_state["analysis"]
        net_disc = a['net_discretionary']

        try:
            import pandas as pd
            import calendar
            from datetime import date
            target         = pd.to_datetime(goal_date, dayfirst=True).date()
            today          = date.today()
            days_left      = (target - today).days
            weeks_left     = days_left / 7
            amount_remaining     = goal_amount
            weekly_contribution  = amount_remaining / weeks_left if weeks_left > 0 else 0
            monthly_contribution = weekly_contribution * 4.33
            budget_remaining     = monthly_budget - net_disc
            days_in_month        = calendar.monthrange(today.year, today.month)[1]
            days_remaining       = days_in_month - today.day
            daily_allowance      = budget_remaining / days_remaining if days_remaining > 0 else 0
        except Exception as e:
            return f"❌ Error in calculation: {e}"

        lines = []
        lines.append("## 📊 Budget & Savings Plan")
        lines.append(f"\n**Monthly Budget:** ${monthly_budget:.2f}")
        lines.append(f"**Spent So Far:** ${net_disc:.2f}")
        lines.append(f"**Budget Remaining:** ${budget_remaining:.2f}")
        lines.append(f"**Daily Allowance Remaining:** ${daily_allowance:.2f}/day")

        if budget_remaining < 0:
            lines.append(
                f"\n⚠️ **You are ${abs(budget_remaining):.2f} over budget this month.**"
            )
        else:
            lines.append(f"\n✅ **You are within budget for this month.**")

        lines.append(f"\n## 🎯 Savings Goal: {goal_name}")
        lines.append(f"- **Target Amount:** ${goal_amount:.2f}")
        lines.append(f"- **Target Date:** {goal_date}")
        lines.append(f"- **Days Remaining:** {days_left}")
        lines.append(f"- **Weekly Saving Needed:** ${weekly_contribution:.2f}")
        lines.append(f"- **Monthly Saving Needed:** ${monthly_contribution:.2f}")

        lines.append("\n## ✂️ Cut X → Save Y Suggestions")
        for cat, row in a['category_summary'].iterrows():
            if cat not in ['Bills & Utilities', 'Income']:
                saving      = row['total'] * 0.5
                weeks_saved = saving / weekly_contribution if weekly_contribution > 0 else 0
                lines.append(
                    f"- Cutting **{cat}** by 50% saves **${saving:.2f}/month** "
                    f"and reaches {goal_name} **{weeks_saved:.1f} weeks sooner**"
                )

        return "\n".join(lines)

    def run_savings_tool(target, monthly, current):
        return savings_calculator(
            f"target={target}, monthly={monthly}, current={current}"
        )

    # ── Tab 4 handlers ────────────────────────────────────────
    def generate_advice():
        if app_state["analysis"] is None:
            return "❌ No data loaded. Please upload a CSV file first."
        return generate_ai_recommendations(app_state["analysis"])

    def chat_with_finn(message, history):
        if app_state["analysis"] is not None:
            a = app_state["analysis"]
            context = (
                f"Student financial data: "
                f"Income ${a['total_income']:.2f}, "
                f"Essential ${a['total_essential']:.2f}, "
                f"Discretionary ${a['net_discretionary']:.2f}, "
                f"Net position ${a['net_position']:.2f}, "
                f"Invisible spend ${a['invisible_total']:.2f}, "
                f"Subscriptions ${a['subscription_total']:.2f}/month, "
                f"Social {a['social_label']}. "
            )
        else:
            context = "No spending data loaded yet. "

        history_text = ""
        for exchange in history[-4:]:
            if isinstance(exchange, dict):
                role    = "Student" if exchange["role"] == "user" else "Finn"
                history_text += f"{role}: {exchange['content']}\n"
            elif isinstance(exchange, (list, tuple)) and len(exchange) == 2:
                history_text += f"Student: {exchange[0]}\nFinn: {exchange[1]}\n"

        prompt = f"""
You are Finn, a friendly financial coach for Australian university students.
Be warm, specific, and concise (3-5 sentences max).
Never suggest cutting essential expenses like rent or utilities.
Always protect a reasonable social budget.

{context}

{history_text}
Student: {message}
Finn:"""

        try:
            response = get_response(prompt)
        except Exception as e:
            response = f"❌ Could not reach AI: {e}"

        return response

    # ── Build the Gradio interface ────────────────────────────
    with gr.Blocks(
        title="💰 Smart Finance Assistant",
        theme=gr.themes.Soft()
    ) as app:

        gr.Markdown("""
# 💰 Smart Finance Assistant
### Your personal financial coach for uni life — find where your money goes and take back control.
""")

        with gr.Tabs():

            # ── Tab 1: Upload & Edit ──────────────────────────
            with gr.Tab("📥 Upload & Edit"):
                gr.Markdown("### Upload your bank CSV or add transactions manually")

                csv_upload    = gr.File(label="Upload Bank CSV", file_types=[".csv"])
                upload_status = gr.Textbox(label="Status", interactive=False)
                transaction_table = gr.Dataframe(
                    label       = "Your Transactions (edit category or essential flag)",
                    interactive = True
                )

                csv_upload.change(
                    fn      = handle_upload_and_display,
                    inputs  = [csv_upload],
                    outputs = [upload_status, transaction_table]
                )

                save_btn    = gr.Button("💾 Save Edits", variant="primary")
                save_status = gr.Textbox(label="Save Status", interactive=False)
                save_btn.click(
                    fn      = handle_table_edit,
                    inputs  = [transaction_table],
                    outputs = [save_status]
                )

                gr.Markdown("### ➕ Add a Cash Transaction")
                with gr.Row():
                    m_date        = gr.Textbox(
                        label       = "Date (DD/MM/YYYY)",
                        placeholder = "23/03/2024"
                    )
                    m_description = gr.Textbox(
                        label       = "Description",
                        placeholder = "Cash - Market Stall"
                    )
                    m_amount = gr.Number(
                        label = "Amount (negative for expense)",
                        value = -20.00
                    )
                with gr.Row():
                    m_category = gr.Dropdown(
                        label   = "Category",
                        choices = [
                            "Food & Groceries", "Coffee & Snacks", "Eating Out",
                            "Transport", "Social", "Subscriptions", "Shopping",
                            "Bills & Utilities", "Other", "Income"
                        ],
                        value = "Other"
                    )
                    m_essential = gr.Checkbox(label="Essential?", value=False)

                add_btn    = gr.Button("➕ Add Transaction", variant="primary")
                add_status = gr.Textbox(label="Status", interactive=False)
                add_btn.click(
                    fn      = handle_manual_entry,
                    inputs  = [m_date, m_description, m_amount, m_category, m_essential],
                    outputs = [add_status, transaction_table]
                )

            # ── Tab 2: Spending Dashboard ─────────────────────
            with gr.Tab("📊 Spending Dashboard"):
                gr.Markdown("### Your spending breakdown at a glance")
                refresh_btn   = gr.Button("🔄 Refresh Dashboard", variant="primary")
                dashboard_out = gr.Markdown()
                refresh_btn.click(
                    fn      = get_dashboard,
                    inputs  = [],
                    outputs = [dashboard_out]
                )

            # ── Tab 3: Budget & Goals ─────────────────────────
            with gr.Tab("🎯 Budget & Goals"):
                gr.Markdown("### Set your budget and track your savings goal")

                monthly_budget = gr.Slider(
                    minimum = 200,
                    maximum = 3000,
                    value   = 800,
                    step    = 50,
                    label   = "Monthly Discretionary Budget ($)"
                )

                gr.Markdown("### 🏖️ Savings Goal")
                with gr.Row():
                    goal_name   = gr.Textbox(
                        label       = "Goal Name",
                        placeholder = "Bali Trip"
                    )
                    goal_amount = gr.Number(
                        label = "Target Amount ($)",
                        value = 2000
                    )
                    goal_date   = gr.Textbox(
                        label       = "Target Date (DD/MM/YYYY)",
                        placeholder = "01/12/2024"
                    )

                plan_btn = gr.Button("📊 Calculate My Plan", variant="primary")
                plan_out = gr.Markdown()
                plan_btn.click(
                    fn      = calculate_budget_plan,
                    inputs  = [monthly_budget, goal_name, goal_amount, goal_date],
                    outputs = [plan_out]
                )

                gr.Markdown("### 💰 Savings Goal Calculator")
                with gr.Row():
                    calc_target  = gr.Number(label="Target Amount ($)",        value=2000)
                    calc_monthly = gr.Number(label="Monthly Contribution ($)",  value=150)
                    calc_current = gr.Number(label="Already Saved ($)",         value=0)

                calc_btn = gr.Button("Calculate", variant="secondary")
                calc_out = gr.Textbox(label="Result", interactive=False, lines=8)
                calc_btn.click(
                    fn      = run_savings_tool,
                    inputs  = [calc_target, calc_monthly, calc_current],
                    outputs = [calc_out]
                )

            # ── Tab 4: AI Advice & Chat ───────────────────────
            with gr.Tab("🤖 AI Advice & Chat"):
                gr.Markdown("### Get personalised financial advice from Finn")

                advice_btn = gr.Button(
                    "✨ Generate My Personalised Advice",
                    variant = "primary"
                )
                advice_out = gr.Markdown()
                advice_btn.click(
                    fn      = generate_advice,
                    inputs  = [],
                    outputs = [advice_out]
                )

                gr.Markdown("### 💬 Chat with Finn")
                gr.ChatInterface(
                    fn       = chat_with_finn,
                    examples = [
                        "Where is most of my money going?",
                        "How can I save for a holiday without giving up going out?",
                        "Are my subscriptions worth keeping?",
                        "I keep overspending after payday — what should I do?"
                    ]
                )

    return app



In [75]:
# ── Launch the app ────────────────────────────────────────────
finance_app = create_finance_assistant_ui()
finance_app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0f0e70138946329e1a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

✅ Loaded 134 transactions successfully.

📊 Data Summary:
   Period:       03 Jan 2025 to 31 Mar 2025
   Income rows:  13
   Expense rows: 121
   Total rows:   134
✅ 9 transactions automatically flagged as essential.

🔒 Essential transactions:
      date              description  amount
2025-01-03 RENT - REALMARK PROPERTY -1050.0
2025-01-09         OPTUS PHONE BILL   -55.0
2025-01-16       SYNERGY POWER BILL   -95.0
2025-02-01 RENT - REALMARK PROPERTY -1050.0
2025-02-09         OPTUS PHONE BILL   -55.0
2025-02-19       SYNERGY POWER BILL   -95.0
2025-03-01 RENT - REALMARK PROPERTY -1050.0
2025-03-09         OPTUS PHONE BILL   -55.0
2025-03-19       SYNERGY POWER BILL   -95.0

  ℹ️  You can manually update the essential column if anything looks wrong.
        💰 SMART FINANCE ASSISTANT SUMMARY

📥  Total Income:            $ 7540.00
🔒  Essential Expenses:      $ 3600.00
💸  Discretionary Spending:  $ 3371.96
💚  Net Position:            $  568.04

────────────────────────────────────────────

---

# 🧪 STEP 6: Test with a Variety of Data

**🔍 Comprehensive Testing Strategy**

Create thorough tests for your Smart Finance Assistant to ensure it handles real-world scenarios.

::: {.callout-tip}
## 🤖 AI Collaboration for Testing

**Effective Testing Prompts:**
```
"Help me create comprehensive test cases for my finance assistant. Include:
- Normal transaction data
- Edge cases (refunds, large amounts, missing data)
- Invalid data scenarios (corrupted files, wrong formats)
- Business logic validation (spending calculations, recommendations)
Create assert statements to verify each scenario."
```
:::

## Foundation Function Tests

In [ ]:
# 🤖 AI Collaboration: Comprehensive Test Suite
# Ask AI to help you create thorough test cases

def create_test_datasets():
    """
    Create various test datasets for comprehensive testing

    🤖 AI Collaboration Prompt:
    "Create realistic test datasets for a finance assistant including:
    1. Normal spending data with various categories
    2. Edge cases: refunds (negative amounts), missing data, zero amounts
    3. Data quality issues: invalid formats, extreme values
    4. Business scenarios: high spending months, savings patterns
    Include Australian business names and realistic amounts."
    """
    # Your AI-generated test data goes here
    pass

def test_data_loading_function():
    """
    Test the data loading and cleaning functionality

    🤖 AI Collaboration Prompt:
    "Create assert statements to test my data loading function with:
    - Valid CSV data
    - CSV with dollar signs in amounts
    - Missing values and invalid data
    - Empty files and corrupted data
    Verify that cleaning works correctly and errors are handled gracefully."
    """
    print("🧪 Testing data loading function...")
    # Your AI-generated test cases go here
    pass

def test_spending_analysis():
    """
    Test spending analysis calculations

    🤖 AI Collaboration Prompt:
    "Create tests for spending analysis that verify:
    - Category totals are calculated correctly
    - Percentages add up to 100%
    - Refunds are handled appropriately
    - Edge cases like single transactions or empty categories
    Use assert statements with known expected results."
    """
    print("🧪 Testing spending analysis...")
    # Your AI-generated analysis tests go here
    pass

def test_business_insights():
    """
    Test business recommendation generation

    🤖 AI Collaboration Prompt:
    "Create tests that verify business insights are appropriate:
    - High spending categories are identified correctly
    - Savings opportunities are realistic
    - Recommendations match spending patterns
    - Output format is user-friendly"
    """
    print("🧪 Testing business insights...")
    # Your AI-generated insight tests go here
    pass

# Run all tests
print("🔍 COMPREHENSIVE TESTING SUITE")
print("=" * 40)

try:
    create_test_datasets()
    test_data_loading_function()
    test_spending_analysis()
    test_business_insights()
    print("✅ All tests passed! Your finance assistant is working correctly.")
except AssertionError as e:
    print(f"❌ Test failed: {e}")
except Exception as e:
    print(f"⚠️ Test error: {e}")

## Advanced Integration Tests

In [ ]:
# 🤖 AI Collaboration: Integration Testing
# Ask AI to help test the complete system integration

def test_full_workflow():
    """
    Test the complete workflow from CSV upload to final recommendations

    🤖 AI Collaboration Prompt:
    "Create an end-to-end test that:
    1. Loads sample CSV data
    2. Runs complete analysis pipeline
    3. Generates chat responses about the data
    4. Verifies RAG system retrieval
    5. Tests custom tool functionality
    Ensure all components work together seamlessly."
    """
    print("🧪 Testing complete workflow integration...")
    # Your AI-generated integration tests go here
    pass

def test_error_handling():
    """
    Test error handling and user experience

    🤖 AI Collaboration Prompt:
    "Create tests that verify error handling for:
    - Invalid file uploads
    - Network connection issues
    - Malformed data
    - User input validation
    Ensure error messages are user-friendly and helpful."
    """
    print("🧪 Testing error handling...")
    # Your AI-generated error tests go here
    pass

# Run integration tests
try:
    test_full_workflow()
    test_error_handling()
    print("✅ Integration tests completed successfully!")
except Exception as e:
    print(f"⚠️ Integration test issue: {e}")

---

# 📊 Project Completion Checklist

## Foundation Skills ✅
- [ ] **Data Processing**: CSV loading and cleaning functions work reliably
- [ ] **Analysis Functions**: Spending summaries calculate correctly
- [ ] **Business Insights**: Recommendations are relevant and actionable  
- [ ] **Error Handling**: Graceful handling of data issues
- [ ] **Testing**: Comprehensive test coverage for core functions
- [ ] **Documentation**: Clear AI collaboration documentation in diary

## Advanced Integration ✅
- [ ] **Chat Interface**: Finance advisor personality implemented
- [ ] **RAG System**: Document retrieval for financial guidance
- [ ] **Custom Tools**: At least one financial calculator/utility
- [ ] **Gradio UI**: Professional, user-friendly interface
- [ ] **Full Integration**: All components work together seamlessly

## Professional Standards ✅
- [ ] **Code Quality**: Professional, commented, maintainable code
- [ ] **Business Focus**: Clear connection to real finance problems
- [ ] **User Experience**: Interface suitable for non-technical users
- [ ] **AI Collaboration**: Extensive, well-documented AI usage
- [ ] **Testing**: Robust validation of all features

## Project Documentation ✅  
- [ ] **Developer's Diary**: Complete AI collaboration documentation
- [ ] **README**: Clear project description and usage instructions
- [ ] **GitHub**: Regular commits showing development progress
- [ ] **Reflection**: Thoughtful analysis of learning and challenges

---

# 🎯 Final Thoughts: Your Finance Assistant Journey

Congratulations on building your Smart Finance Assistant! This project represents a significant achievement in modern business programming:

**Technical Skills Developed:**
- AI-assisted development workflows
- Professional data processing with pandas
- Integration of multiple AI technologies
- User interface design with Gradio
- Comprehensive software testing

**Business Skills Developed:**  
- Financial data analysis and insights
- User-centered application design
- Professional documentation practices
- Iterative development methodology
- Critical evaluation of AI suggestions

**Professional Preparation:**
- Experience with industry-standard AI collaboration
- Portfolio-ready application development
- Understanding of business problem-solving with technology
- Documentation practices for workplace environments

**Your Smart Finance Assistant demonstrates your ability to direct AI tools toward meaningful business solutions - exactly the skill set that modern BIS graduates need for career success!**

---

*Remember to document all AI collaborations in your Developer's Diary and maintain regular GitHub commits throughout your development process.*
